# Artificial Intelligence - Exercise 1
## International Football Results (1872–2026) Analysis

This notebook performs a comprehensive analysis of international football matches, exploring goal trends, home advantage, and historical team performance using the Kaggle International Football Results dataset.

### Step 1: Load the CSV

We start by importing the necessary libraries and loading the dataset into a Pandas DataFrame.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic parameters for premium visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

df = pd.read_csv("results.csv")
df.head()

### Basic Exploration

**Logic:** We utilize built-in Pandas methods like `.shape` for counting records, `.min()`/`.max()` on the date column to determine the timeframe, and `.nunique()` to identify the diversity of locations. We also use `.value_counts()` to find the most active teams.

In [ ]:
# 1. How many matches are in the dataset?
print(f"Total matches recorded: {df.shape[0]:,}")

# 2. What is the earliest and latest year in the data?
df['date'] = pd.to_datetime(df['date'])
print(f"Earliest year in data: {df['date'].dt.year.min()}")
print(f"Latest year in data: {df['date'].dt.year.max()}")

# 3. How many unique countries are there?
unique_countries = df['country'].nunique()
print(f"Number of unique hosting countries: {unique_countries}")

# 4. Which team appears most frequently as home team?
print("\nTop 5 teams appearing as home team:")
print(df["home_team"].value_counts().head())

### Goals Analysis

**Logic:** To analyze goal trends, we compute a new feature `total_goals` which is the sum of `home_score` and `away_score`. This enables us to calculate the average scoring rate and identify anomalous high-scoring matches.

In [ ]:
# Create total goals feature
df["total_goals"] = df["home_score"] + df["away_score"]

# 5. What is the average number of goals per match?
avg_goals = df["total_goals"].mean()
print(f"Average goals per match: {avg_goals:.2f}")

# 6. What is the highest scoring match?
highest_match = df.loc[df["total_goals"].idxmax()]
print(f"\nHighest scoring match: {highest_match['home_team']} vs {highest_match['away_team']}")
print(f"Score: {highest_match['home_score']} - {highest_match['away_score']} (Total: {highest_match['total_goals']})")

# 7. Are more goals scored at home or away?
home_goals = df['home_score'].sum()
away_goals = df['away_score'].sum()
print(f"\nTotal Home Goals: {home_goals:,}")
print(f"Total Away Goals: {away_goals:,}")
print("Observation: More goals are scored by the home team.")

# 8. What is the most common total goals value?
most_common_goals = df["total_goals"].mode()[0]
print(f"\nMost common total goals in a match: {most_common_goals}")

### Match Results

**Logic:** We categorize every match into 'Home Win', 'Away Win', or 'Draw' using a custom function applied across the rows. This categorization is crucial for quantifying 'Home Advantage' and tracking individual team success.

In [ ]:
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    else:
        return "Draw"

df["result"] = df.apply(match_result, axis=1)

# 9. What percentage of matches are home wins?
home_win_pct = (df["result"] == "Home Win").mean() * 100
print(f"Percentage of matches that are home wins: {home_win_pct:.2f}%")

# 10. Does home advantage exist?
away_win_pct = (df["result"] == "Away Win").mean() * 100
print(f"Home Win Rate: {home_win_pct:.2f}% | Away Win Rate: {away_win_pct:.2f}%")
print("Conclusion: Yes, home advantage is statistically significant.")

# 11. Which country has the most wins historically?
def get_winner(row):
    if row["home_score"] > row["away_score"]: return row["home_team"]
    elif row["home_score"] < row["away_score"]: return row["away_team"]
    else: return None

df["winner"] = df.apply(get_winner, axis=1)
most_wins = df["winner"].value_counts().idxmax()
print(f"\nTeam with most historical wins: {most_wins}")

### Visualization

**Logic:** Visualizations provide a clear perspective on the data distributions. We use a histogram for goal counts, a bar chart for categorical results, and a horizontal bar chart to rank historical leaders.

In [ ]:
# 1. Histogram of goals
plt.figure(figsize=(10, 6))
sns.histplot(df["total_goals"], bins=30, kde=True, color="#4a90e2")
plt.title("Distribution of Total Goals Per Match", fontsize=16, fontweight='bold')
plt.xlabel("Total Goals")
plt.ylabel("Frequency")
plt.xlim(0, 15)
plt.show()

# 2. Bar chart of match outcomes
plt.figure(figsize=(10, 6))
sns.countplot(x="result", data=df, palette="viridis", order=["Home Win", "Away Win", "Draw"])
plt.title("Historical Match Outcomes", fontsize=16, fontweight='bold')
plt.xlabel("Outcome")
plt.ylabel("Match Count")
plt.show()

# 3. Top 10 teams by total wins
plt.figure(figsize=(10, 6))
top_wins = df["winner"].value_counts().head(10)
sns.barplot(x=top_wins.values, y=top_wins.index, palette="magma")
plt.title("Top 10 Teams by Total Historical Wins", fontsize=16, fontweight='bold')
plt.xlabel("Number of Wins")
plt.ylabel("Team")
plt.show()